# Giáo trình Dữ liệu lớn – Chương 3

Notebook tổng hợp các đoạn mã trong chương, chạy trên **Google Colab** (ô đầu cài OpenJDK 17, PySpark 3.5.7 và tải kho mã). Trên **Databricks Free Edition**: bỏ ô cài đặt, tải thư mục `data/` lên volume `/Volumes/workspace/default/du_lieu/` do người học tự tạo và thay `data/` bằng đường dẫn này; tính toán serverless của Free Edition không hỗ trợ API RDD/`SparkContext` và `cache()`/`persist()` (xem [README](https://github.com/mocminh/bigdata_code#databricks-free-edition)).

Khác biệt so với sách: đường dẫn `hdfs://.../data/` được đổi thành `data/`, thư mục ghi kết quả là `output/` và `models/`; lệnh `spark.stop()` được đổi thành ghi chú để các ô sau vẫn chạy được. Mã nguyên văn: `code/ch03/doan_ma_*.py`.


In [ ]:
# --- Chuan bi moi truong (Google Colab) ---
# Buoc 1: cai OpenJDK 17 (Spark 3.5 ho tro Java 8/11/17)
!apt-get update -qq
!apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
# Buoc 2: cai dat PySpark tu PyPI (ghim phien ban theo Bang 2.3)
!pip install -q pyspark==3.5.7
if not os.path.exists("data"):
    !git clone -q https://github.com/mocminh/bigdata_code
    %cd bigdata_code
import shutil
for thu_muc in ("output", "models"):          # don ket qua cua lan chay truoc
    shutil.rmtree(thu_muc, ignore_errors=True)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (SparkSession.builder.master("local[*]")
         .appName("GiaoTrinhDuLieuLon-ch03").getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark", spark.version)

## Đoạn mã 3.1. Khởi tạo SparkContext và tạo RDD bằng parallelize, textFile.


In [ ]:
from pyspark.sql import SparkSession
# Khoi tao SparkSession o che do local voi 4 luong xu ly
spark = (SparkSession.builder
         .appName("Chuong3-RDD")
         .master("local[4]")
         .getOrCreate())
sc = spark.sparkContext
# Cach 1: phan tan mot collection co san tren Driver
diem_thi = [7.5, 8.0, 6.5, 9.0, 5.5, 8.5, 7.0, 6.0]
rdd_diem = sc.parallelize(diem_thi, numSlices=4)
# Cach 2: doc du lieu tu he thong luu tru ngoai
rdd_log = sc.textFile("data/access.log")
rdd_cucbo = sc.textFile("data/vidu.txt")
print(rdd_diem.getNumPartitions())   # ket qua: 4
print(rdd_diem.take(3))              # [7.5, 8.0, 6.5]

## Đoạn mã 3.2. Kiểm tra và điều chỉnh số partition của RDD.


In [ ]:
rdd_a = sc.parallelize(range(1, 100001))    # dung gia tri mac dinh
print(rdd_a.getNumPartitions())             # thuong = so loi CPU
rdd_b = sc.parallelize(range(1, 100001), 8) # chi dinh 8 partition
# repartition: tang/giam so partition, gay shuffle toan bo du lieu
rdd_16 = rdd_b.repartition(16)
# coalesce: giam so partition bang cach gop, tranh duoc shuffle
rdd_4 = rdd_16.coalesce(4)
print(rdd_16.getNumPartitions(), rdd_4.getNumPartitions())   # 16 4

## Đoạn mã 3.3. Các phép biến đổi cơ bản trên RDD.


In [ ]:
cau_tho = ["gio theo loi gio may duong may",
           "dong nuoc buon thiu hoa bap lay",
           "thuyen ai dau ben song trang do"]
rdd_cau = sc.parallelize(cau_tho, 2)
# map: moi cau cho ra so tu cua cau do (quan he mot-mot)
rdd_sotu = rdd_cau.map(lambda c: len(c.split()))
# flatMap: moi cau sinh ra nhieu tu roi lam phang thanh mot RDD
rdd_tu = rdd_cau.flatMap(lambda c: c.split())
# filter: giu lai cac tu co it nhat 4 ky tu
rdd_tu_dai = rdd_tu.filter(lambda t: len(t) >= 4)
# distinct: loai bo tu trung lap (co xao tron du lieu)
rdd_duynhat = rdd_tu.distinct()
# union: hop voi mot RDD khac cung kieu
rdd_hop = rdd_tu.union(sc.parallelize(["song", "nui"]))
# sample: lay mau ngau nhien khoang 30% so tu, khong hoan lai
rdd_mau = rdd_tu.sample(withReplacement=False,
                        fraction=0.3, seed=42)
print(rdd_sotu.collect())    # [7, 7, 7]

## Đoạn mã 3.4. Chuỗi biến đổi chỉ được thực thi khi gặp action.


In [ ]:
# Ba dong duoi day hoan tat tuc thi, ke ca voi tep hang tram GB,
# vi chua co bat ky tinh toan nao duoc thuc hien
rdd_log = sc.textFile("data/access.log")
rdd_loi = rdd_log.filter(lambda dong: "ERROR" in dong)
rdd_thongdiep = rdd_loi.map(lambda dong: dong.split("\t")[2])
# In pha he (lineage) de quan sat chuoi phu thuoc giua cac RDD
print(rdd_thongdiep.toDebugString().decode("utf-8"))
# Action moi la thoi diem Spark xay DAG, toi uu va thuc thi
so_loi = rdd_thongdiep.count()
nam_dong_dau = rdd_thongdiep.take(5)   # chi quet du lieu vua du

## Đoạn mã 3.5. Các action thông dụng trên RDD.


In [ ]:
rdd_diem = sc.parallelize([7.5, 8.0, 6.5, 9.0, 5.5, 8.5], 3)
print(rdd_diem.count())      # 6: tong so phan tu
print(rdd_diem.first())      # 7.5: phan tu dau tien
print(rdd_diem.take(4))      # [7.5, 8.0, 6.5, 9.0]
print(rdd_diem.collect())    # keo TOAN BO du lieu ve Driver
# reduce voi ham cong: tinh tong roi suy ra diem trung binh
tong = rdd_diem.reduce(lambda a, b: a + b)
print("Diem trung binh:", tong / rdd_diem.count())
# Ghi ket qua xuong HDFS, moi partition mot tep part-xxxxx
rdd_diem.saveAsTextFile("output/diem")
# foreach chay ngay tai Executor; print nam trong log Executor
rdd_diem.foreach(lambda x: print(x))

## Đoạn mã 3.6. Đo khác biệt thời gian trước và sau khi cache RDD.


In [ ]:
import time
from pyspark import StorageLevel

rdd_gd = (sc.textFile("data/giaodich.csv")
          .map(lambda dong: dong.split(","))
          .filter(lambda tr: tr[3] == "THANH CONG"))
t0 = time.time()
print(rdd_gd.count())              # lan 1: doc va loc lai tu nguon
print("Chua cache: %.2f giay" % (time.time() - t0))
rdd_gd.persist(StorageLevel.MEMORY_AND_DISK)
rdd_gd.count()                     # action nap du lieu vao cache
t1 = time.time()
print(rdd_gd.count())              # doc truc tiep tu bo nho dem
print("Da cache:   %.2f giay" % (time.time() - t1))
rdd_gd.unpersist()                 # giai phong khi khong dung nua

## Đoạn mã 3.7. So sánh groupByKey và reduceByKey trên cùng bài toán.


In [ ]:
rdd_cap = sc.parallelize(
    [("spark", 1), ("hadoop", 1), ("spark", 1),
     ("kafka", 1), ("spark", 1), ("hadoop", 1)], 2)
# Cach 1: groupByKey keo toan bo gia tri cua moi khoa qua mang
kq1 = (rdd_cap.groupByKey()
              .mapValues(lambda ds: sum(ds))
              .collect())
# Cach 2: reduceByKey gop cuc bo o tung partition truoc khi shuffle
kq2 = rdd_cap.reduceByKey(lambda a, b: a + b).collect()
# Hai cach cho cung ket qua nhung chi phi shuffle rat khac nhau
print(sorted(kq1) == sorted(kq2))   # True

## Đoạn mã 3.8. Chương trình đếm từ hoàn chỉnh với RDD.


In [ ]:
rdd_vanban = sc.textFile("data/vanban.txt")
rdd_demtu = (rdd_vanban
    .flatMap(lambda dong: dong.lower().split())  # tach thanh tu
    .map(lambda tu: (tu, 1))                     # cap (tu, 1)
    .reduceByKey(lambda a, b: a + b))            # cong theo khoa
# Lay 10 tu pho bien nhat: doi (tu, dem) thanh (dem, tu) roi sap xep
top10 = (rdd_demtu
    .map(lambda cap: (cap[1], cap[0]))
    .sortByKey(ascending=False)
    .take(10))
for dem, tu in top10:
    print("%-15s %d" % (tu, dem))
rdd_demtu.saveAsTextFile("output/demtu")

## Đoạn mã 3.9. Kết nối hai tập dữ liệu sinh viên và điểm thi.


In [ ]:
# (ma sinh vien, ho ten)
rdd_sv = sc.parallelize([("SV01", "Nguyen Van An"),
                         ("SV02", "Tran Thi Binh"),
                         ("SV03", "Le Van Cuong"),
                         ("SV04", "Pham Thi Dung")])
# (ma sinh vien, diem mon Du lieu lon)
rdd_diem = sc.parallelize([("SV01", 8.5), ("SV02", 7.0),
                           ("SV01", 9.0), ("SV05", 6.5)])
# join (ket noi trong): chi giu khoa co mat o CA HAI phia
print(rdd_sv.join(rdd_diem).collect())
# leftOuterJoin: giu ca sinh vien chua co diem (gia tri None)
print(rdd_sv.leftOuterJoin(rdd_diem).collect())
# cogroup: moi khoa kem hai day gia tri tu hai RDD
gom = rdd_sv.cogroup(rdd_diem)
print([(k, (list(a), list(b))) for k, (a, b) in gom.collect()])
# countByKey: dem so ban ghi theo khoa, tra ve dict tai Driver
print(dict(rdd_diem.countByKey()))
# ket qua: {'SV01': 2, 'SV02': 1, 'SV05': 1}

## Đoạn mã 3.10. Dữ liệu mẫu cho bài tập đếm từ.


In [ ]:
du_lieu = [
    "spark xu ly du lieu lon rat nhanh",
    "rdd la nen tang cot loi cua spark",
    "du lieu lon doi hoi xu ly phan tan",
    "spark danh gia tre cac phep bien doi tren du lieu"
]

## Đoạn mã 3.11. Dữ liệu mẫu cho bài tập Pair RDD.


In [ ]:
don_hang = [("KH01", 250.0), ("KH02", 120.0), ("KH01", 380.0),
            ("KH03", 90.0), ("KH02", 60.0), ("KH01", 45.0)]
khach_hang = [("KH01", "Nguyen Van An"), ("KH02", "Tran Thi Binh"),
              ("KH03", "Le Van Cuong"), ("KH04", "Pham Thi Dung")]

## Đoạn mã 3.12. Hai nhánh phân tích dùng chung một RDD trung gian.


> Khung Bài 3.4 giả định đã có hàm `chuan_hoa` – ô chuẩn bị bên dưới định nghĩa một hàm mẫu để đoạn mã chạy được.


In [ ]:
# Chuan bi: ham chuan_hoa mau (lam sach mot dong log)
def chuan_hoa(dong):
    return " ".join(dong.replace("\t", " ").split())

In [ ]:
# chuan_hoa: ham lam sach mot dong log, chi phi tinh toan lon
logs = sc.textFile("data/access.log")
errs = logs.filter(lambda d: "ERROR" in d).map(chuan_hoa)

so_loi   = errs.count()
theo_ip  = errs.map(lambda d: (d.split(" ")[0], 1)) \
               .reduceByKey(lambda a, b: a + b).collect()